
# 🗽 NYC YELLOW TAXI DATA PIPELINE 🚖

---

Welcome to the NYC Taxi ETL pipeline notebook.<br>
This pipeline performs a robust transformation and loading process of the **Yellow Taxi** dataset as part of the NYCTAXI project.

---

**Pipeline Overview:**
1. **Source Table:** `NYCTAXI.BRONZE.YELLOW_TAXI`
2. **Key Steps:**
   - Read source data
   - Explore and validate timestamps
   - Filter for first half of 2026
   - Decode and transform key attributes
   - Save cleansed records to **Silver Table**
3. **Target Table:** `NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED`

---

> *Follow the steps below for clean, organized Yellow Taxi trip data ready for analytics.*

#### READ FROM INGESTION TABLE
- NYCTAXI.BRONZE.YELLOW_TAXI

In [0]:
nyc_yellow_taxi_df = spark.read.table('NYCTAXI.BRONZE.YELLOW_TAXI')
nyc_yellow_taxi_df.limit(3).display()

In [0]:
from pyspark.sql.functions import *

val_df = nyc_yellow_taxi_df.agg(max('tpep_pickup_datetime').alias('max_pickup_datetime'), 
                                min('tpep_pickup_datetime').alias('min_pickup_datetime')).display()

In [0]:
from pyspark.sql.functions import *

nyc_yellow_taxi_filtered_trans_df = nyc_yellow_taxi_df.filter((col("tpep_pickup_datetime") >= "2025-01-01") & (col("tpep_pickup_datetime") < "2026-07-01"))

In [0]:
from pyspark.sql.functions import col, when, unix_timestamp

nyc_yellow_taxi_trans_df = nyc_yellow_taxi_filtered_trans_df.select(
    when(col("VendorID") == 1, "Creative Mobile Technologies, LLC")
      .when(col("VendorID") == 2, "Curb Mobility, LLC")
      .when(col("VendorID") == 6, "Myle Technologies Inc")
      .when(col("VendorID") == 7, "Helix")
      .otherwise("Unknown")
      .alias("vendor"),
    
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    # Calculate trip duration in minutes
    (
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60
    ).alias("trip_duration"),
    "passenger_count",
    "trip_distance",

    when(col("RatecodeID") == 1, "Standard Rate")
      .when(col("RatecodeID") == 2, "JFK")
      .when(col("RatecodeID") == 3, "Newark")
      .when(col("RatecodeID") == 4, "Nassau or Westchester")
      .when(col("RatecodeID") == 5, "Negotiated Fare")
      .when(col("RatecodeID") == 6, "Group Ride")
      .otherwise("Unknown")
      .alias("rate_type"),
    
    "store_and_fwd_flag",
    col("PULocationID").alias("pu_location_id"),
    col("DOLocationID").alias("do_location_id"),
    
    when(col("payment_type") == 0, "Flex Fare trip")
      .when(col("payment_type") == 1, "Credit card")
      .when(col("payment_type") == 2, "Cash")
      .when(col("payment_type") == 3, "No charge")
      .when(col("payment_type") == 4, "Dispute")
      .when(col("payment_type") == 6, "Voided trip")
      .otherwise("Unknown")
      .alias("payment_type"),
    
    "fare_amount",
    "extra",
    "mta_tax",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    col("Airport_fee").alias("airport_fee"),
    "cbd_congestion_fee",
    "load_timestamp"
)

# nyc_yellow_taxi_trans_df.display(10)

#### LOAD THE DATA INTO THE SILVER TABLE
- NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED

In [0]:
nyc_yellow_taxi_trans_df.write.mode('overwrite').saveAsTable('NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED')

In [0]:
dbutils.notebook.exit('YELLOW TAXI TRIP HAS BEEN LOADED INTO NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_CLEANSED')